In [1]:
INPUT_CSV = "dataset/order_brush_order.csv"
OUTPUT_CSV = "result/result.csv"

# Import Data

In [2]:
import pandas

raw_data = pandas.read_csv(INPUT_CSV)

In [3]:
from datetime import datetime


def get_unix(s):
    _temp = datetime.strptime(s, "%Y-%m-%d %H:%M:%S")
    return int((_temp - datetime(1970,1,1)).total_seconds())

In [4]:
clean_data = raw_data.copy(deep = True)

clean_data["event_time_unix"] = clean_data.event_time.apply(get_unix)

# Get Shop

In [5]:
distinct_shop = clean_data["shopid"].drop_duplicates().reset_index(drop=True)

shop_data = [
    {
        "id": id,
        "brush": dict(),
        "result": set(),
        "print": "0",
        "table": clean_data[clean_data.shopid == id][[
                "orderid", "userid", "event_time_unix"
            ]].sort_values(
                by=["event_time_unix"]
            ),
    }
    for id in distinct_shop
]

# Process Each Shop

In [6]:
for shop in shop_data:
    n = shop["table"].shape[0]

    for left in range(n):
        for right in range(left,n):
            if shop["table"].iloc[right,2] - shop["table"].iloc[left,2] <= 3600:
                is_okay = True
                
                if 0 <= left-1 and right+1 < n:
                    is_okay = (shop["table"].iloc[right+1,2] - shop["table"].iloc[left-1,2] >= 3600)
                    
                if is_okay:
                    temp_data = shop["table"][left:right+1]

                    # analyze
                    rate = len(temp_data) / len(temp_data["userid"].unique())
                    if rate >= 3.0:
                        temp_len = len(temp_data)

                        for idx in range(temp_len):
                            data = temp_data.iloc[idx]
                            user_id = int(data["userid"])
                            if not user_id in shop["brush"]:
                                shop["brush"][int(user_id)] = set()

                            shop["brush"][user_id].add(data["orderid"])
            else:
                break

In [7]:
for shop in shop_data:
    total = sum([ len(v) for v in shop["brush"].values() ])
    
    if total > 0:
        maxx = max([ len(v) for v in shop["brush"].values() ])

        for k,v in shop["brush"].items():
            if len(v) == maxx:
                shop["result"].add(int(k))
        
        if len(shop["result"]) > 0:
            rett = list(shop["result"])
            rett.sort()

            for x in rett:
                shop["print"] += "&{0}".format(x)

            shop["print"] = shop["print"][2:]

# Save Result

In [8]:
result = pandas.DataFrame({
    "shopid": [ shop["id"] for shop in shop_data ],
    "userid": [ shop["print"] for shop in shop_data ],
})

In [9]:
result.to_csv(OUTPUT_CSV, index=False)